# 🐦 Flappy Bird DQN Training (Google Colab)

This notebook trains a Deep Q-Network (DQN) agent with 15-ray LIDAR obstacle detection.
Google Colab has **PyTorch pre-installed** and ultra-fast internet.

**Workflow:**
1. Run all cells below (trains in ~10-15 minutes for 10,000–20,000 episodes).
2. Download `model_best.pth`.
3. Run locally: `python demo.py` to watch your AI play with the red rays!

In [ ]:
# 1. Install headless pygame
!pip install pygame-ce -q

In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"  # Headless mode for Google Colab

import math
import random
import time
from collections import deque
import pygame
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import files

print(f"Using PyTorch {torch.__version__} on device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# 2. Environment with 15 LIDAR Rays
SCREEN_WIDTH = 288
SCREEN_HEIGHT = 512
PIPE_GAP = 100
PIPE_WIDTH = 52
BIRD_WIDTH = 34
BIRD_HEIGHT = 24
GRAVITY = 0.8
FLAP_STRENGTH = -8.0
PIPE_SPEED = 4
NUM_RAYS = 15
RAY_ANGLES = np.linspace(-math.pi / 2.5, math.pi / 2.5, NUM_RAYS)
MAX_RAY_DIST = 350.0

class FlappyBirdEnv:
    def __init__(self):
        pygame.init()
        self.reset()

    def reset(self):
        self.bird_x = 50.0
        self.bird_y = float(SCREEN_HEIGHT // 2)
        self.bird_vel = 0.0
        self.pipes = []
        self._spawn_pipe(SCREEN_WIDTH + 50)
        self._spawn_pipe(SCREEN_WIDTH + 50 + 200)
        self.score = 0
        self.ray_distances = [1.0] * NUM_RAYS
        return self._get_state()

    def _spawn_pipe(self, x_pos):
        gap_y = random.randint(120, SCREEN_HEIGHT - 120)
        top_pipe = pygame.Rect(x_pos, 0, PIPE_WIDTH, gap_y - PIPE_GAP // 2)
        bottom_pipe = pygame.Rect(x_pos, gap_y + PIPE_GAP // 2, PIPE_WIDTH, SCREEN_HEIGHT - (gap_y + PIPE_GAP // 2))
        self.pipes.append({"top": top_pipe, "bottom": bottom_pipe, "gap_y": gap_y, "passed": False})

    def _cast_rays(self):
        bird_center = (self.bird_x + BIRD_WIDTH / 2, self.bird_y + BIRD_HEIGHT / 2)
        ray_distances = []
        all_obstacles = [p["top"] for p in self.pipes] + [p["bottom"] for p in self.pipes] \
                        + [pygame.Rect(0, SCREEN_HEIGHT - 20, SCREEN_WIDTH, 40), pygame.Rect(0, -20, SCREEN_WIDTH, 20)]

        for angle in RAY_ANGLES:
            cos_a = math.cos(angle)
            sin_a = math.sin(angle)
            closest_dist = MAX_RAY_DIST
            target = (bird_center[0] + cos_a * MAX_RAY_DIST, bird_center[1] + sin_a * MAX_RAY_DIST)
            for rect in all_obstacles:
                clipped = rect.clipline(bird_center, target)
                if clipped:
                    dist = math.hypot(clipped[0][0] - bird_center[0], clipped[0][1] - bird_center[1])
                    if dist < closest_dist:
                        closest_dist = dist
            ray_distances.append(closest_dist / MAX_RAY_DIST)
        self.ray_distances = ray_distances
        return ray_distances

    def _get_state(self):
        ray_dists = self._cast_rays()
        next_pipe = next((p for p in self.pipes if p["top"].right > self.bird_x), None)
        dx = (next_pipe["top"].centerx - self.bird_x) / SCREEN_WIDTH if next_pipe else 1.0
        dy_gap = (next_pipe["gap_y"] - self.bird_y) / SCREEN_HEIGHT if next_pipe else 0.0
        norm_bird_y = self.bird_y / SCREEN_HEIGHT
        norm_vel = self.bird_vel / 15.0
        return np.array(ray_dists + [norm_bird_y, norm_vel, dx, dy_gap], dtype=np.float32)

    def step(self, action):
        if action == 1:
            self.bird_vel = FLAP_STRENGTH
        self.bird_vel = min(self.bird_vel + GRAVITY, 12.0)
        self.bird_y += self.bird_vel
        for p in self.pipes:
            p["top"].x -= PIPE_SPEED
            p["bottom"].x -= PIPE_SPEED
        if self.pipes and self.pipes[0]["top"].right < 0:
            self.pipes.pop(0)
        if self.pipes[-1]["top"].centerx < SCREEN_WIDTH - 150:
            self._spawn_pipe(SCREEN_WIDTH + 40)

        bird_rect = pygame.Rect(int(self.bird_x), int(self.bird_y), BIRD_WIDTH, BIRD_HEIGHT)
        done = False
        reward = 0.1
        if self.bird_y <= 0 or self.bird_y + BIRD_HEIGHT >= SCREEN_HEIGHT - 20:
            done = True
            reward = -5.0
        for p in self.pipes:
            if bird_rect.colliderect(p["top"]) or bird_rect.colliderect(p["bottom"]):
                done = True
                reward = -5.0
                break
            if not p["passed"] and p["top"].centerx < self.bird_x:
                p["passed"] = True
                self.score += 1
                reward = 10.0

        next_state = self._get_state()
        if min(self.ray_distances) < 0.1 and not done:
            reward -= 0.2
        return next_state, reward, done, {"score": self.score}

In [ ]:
# 3. DQN Architecture & Replay Buffer
class QNetwork(nn.Module):
    def __init__(self, state_dim=19, action_dim=2):
        super(QNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )
    def forward(self, x):
        return self.net(x)

class ReplayBuffer:
    def __init__(self, capacity=60000):
        self.buffer = deque(maxlen=capacity)
    def push(self, s, a, r, ns, d):
        self.buffer.append((s, a, r, ns, d))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (
            torch.tensor(np.array(s), dtype=torch.float32),
            torch.tensor(a, dtype=torch.int64).unsqueeze(1),
            torch.tensor(r, dtype=torch.float32).unsqueeze(1),
            torch.tensor(np.array(ns), dtype=torch.float32),
            torch.tensor(d, dtype=torch.float32).unsqueeze(1),
        )
    def __len__(self):
        return len(self.buffer)

In [ ]:
# 4. Training Loop (Colab GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = FlappyBirdEnv()
state_dim = 19
action_dim = 2

policy_net = QNetwork(state_dim, action_dim).to(device)
target_net = QNetwork(state_dim, action_dim).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=5e-4)
loss_fn = nn.SmoothL1Loss()
replay_buffer = ReplayBuffer(60000)

NUM_EPISODES = 15000
BATCH_SIZE = 64
GAMMA = 0.99
EPSILON = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 0.9997
TARGET_UPDATE = 500

step_count = 0
best_score = 0
recent_scores = deque(maxlen=100)
all_scores = []

print(f"[*] Starting training for {NUM_EPISODES} episodes on {device}...")
start_time = time.time()

for ep in range(1, NUM_EPISODES + 1):
    state = env.reset()
    done = False
    while not done:
        if random.random() < EPSILON:
            action = random.randint(0, 1)
        else:
            with torch.no_grad():
                st = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                action = policy_net(st).argmax(dim=1).item()

        next_state, reward, done, info = env.step(action)
        replay_buffer.push(state, action, reward, next_state, done)
        state = next_state

        if len(replay_buffer) >= BATCH_SIZE:
            s_b, a_b, r_b, ns_b, d_b = replay_buffer.sample(BATCH_SIZE)
            s_b, a_b, r_b, ns_b, d_b = s_b.to(device), a_b.to(device), r_b.to(device), ns_b.to(device), d_b.to(device)
            q_curr = policy_net(s_b).gather(1, a_b)
            with torch.no_grad():
                q_max = target_net(ns_b).max(1)[0].unsqueeze(1)
                q_targ = r_b + (1 - d_b) * GAMMA * q_max
            loss = loss_fn(q_curr, q_targ)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
            optimizer.step()
            step_count += 1
            if step_count % TARGET_UPDATE == 0:
                target_net.load_state_dict(policy_net.state_dict())
            if EPSILON > EPSILON_END:
                EPSILON *= EPSILON_DECAY

    score = info["score"]
    recent_scores.append(score)
    all_scores.append(score)
    if score > best_score:
        best_score = score
        torch.save(policy_net.state_dict(), "model_best.pth")

    if ep % 200 == 0:
        print(f"Ep: {ep:5d} | Avg Score (100): {np.mean(recent_scores):5.2f} | Max Score: {best_score:3d} | Epsilon: {EPSILON:5.3f} | Time: {time.time()-start_time:5.1f}s")

print("[+] Training Finished! Exporting weights...")
import json
# Export both PyTorch .pth and lightweight JSON weights for pure NumPy
torch.save(policy_net.state_dict(), "model_best.pth")
weights_dict = {k: v.cpu().numpy().tolist() for k, v in policy_net.state_dict().items()}
with open("model_weights.json", "w") as f:
    json.dump(weights_dict, f)
files.download("model_weights.json")
files.download("model_best.pth")
